# 🚀 Notebook 09 — Model Export & Serving

## 🎯 Objective
Package the **best-performing model** (preprocessing + estimator) into a portable artifact and expose a
**predict() interface** suitable for batch and real-time inference. We’ll also build a minimal **FastAPI**
service and a simple **Streamlit** demo for interactive testing.

## 🔗 Inputs
- Best model pointer: `reports/best_model.json`
- Model artifact: `models/heart_model_<run_id>.pkl`
- Schema: feature set used in training (must match pipeline expectations)

## 📦 Outputs
- `models/heart_model_<run_id>.pkl` (validated)
- `models/model_manifest_<run_id>.json` (metadata & schema)
- `src/serve.py` (predict utilities)
- `app/api.py` (FastAPI app)
- `app/streamlit_app.py` (Streamlit demo)

## 🧩 What we’ll do
1. Load **best model** and validate on sample rows
2. Define canonical **inference schema** and input checks
3. Implement **predict_proba / predict** helpers
4. Save a **model manifest** (version, features, run_id)
5. Provide **FastAPI** service + **health/predict** endpoints
6. Provide **Streamlit** demo UI for manual testing


## 0) Setup & import

In [2]:
# (Colab only) mount & cd
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction



Mounted at /content/drive
/content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction


In [3]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
sys.path.append(os.getcwd())
!pip install -q pyngrok

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from src.data import PROC_DIR
MODELS_DIR  = Path("models");  MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)
APP_DIR     = Path("app");     APP_DIR.mkdir(parents=True, exist_ok=True)

1) Load test split (sanity data) & best model artifact

In [4]:
# Data for quick smoke tests
X_test = pd.read_csv(PROC_DIR / "X_test.csv")
y_test = pd.read_csv(PROC_DIR / "y_test.csv").squeeze("columns")

# Resolve best run_id (written in 05 or 07)
best_meta_path = REPORTS_DIR / "best_model.json"
if best_meta_path.exists():
    best_meta = json.loads(best_meta_path.read_text())
    run_id = best_meta["best_run_id"]
else:
    # fallback: newest artifact
    candidates = sorted(MODELS_DIR.glob("heart_model_*.pkl"), key=lambda p: p.stat().st_mtime, reverse=True)
    assert candidates, "No model artifacts found in models/."
    run_id = candidates[0].stem.replace("heart_model_", "")

artifact_path = MODELS_DIR / f"heart_model_{run_id}.pkl"
print("Using model:", artifact_path.name)
pipe = joblib.load(artifact_path)


Using model: heart_model_logreg_tuned_seed42_1761367619.pkl


## 2).  Define the inference schema (feature order & dtypes)

In [5]:
# Canonical raw feature schema (columns expected by pipeline BEFORE derivations)
FEATURES_ORDER = [
    "age","sex","cp","trestbps","chol","fbs","restecg",
    "thalach","exang","oldpeak","slope","ca","thal"
]
DTYPES = {
    "age": float, "sex": int, "cp": int, "trestbps": float, "chol": float,
    "fbs": int, "restecg": int, "thalach": float, "exang": int,
    "oldpeak": float, "slope": int, "ca": float, "thal": int
}


## 3). Build validation & prediction helpers (portable)

In [6]:
def validate_input_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure input DataFrame has all required columns in proper order & dtypes."""
    missing = [c for c in FEATURES_ORDER if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    df = df[FEATURES_ORDER].copy()
    for c, t in DTYPES.items():
        try:
            df[c] = df[c].astype(t)
        except Exception:
            raise TypeError(f"Column '{c}' cannot be cast to {t}")
    return df

def predict_proba_frame(model_pipe, df: pd.DataFrame) -> np.ndarray:
    dfv = validate_input_frame(df)
    return model_pipe.predict_proba(dfv)[:, 1]

def predict_label_frame(model_pipe, df: pd.DataFrame, threshold: float = 0.5) -> np.ndarray:
    proba = predict_proba_frame(model_pipe, df)
    return (proba >= threshold).astype(int)


## 4) Smoke test on a few rows

In [7]:
sample = X_test[FEATURES_ORDER].head(5).copy()
p = predict_proba_frame(pipe, sample)
yhat = (p >= 0.5).astype(int)
pd.DataFrame({"proba": p, "pred": yhat})


,proba,pred
0,0.328690,0
1,0.434443,0
2,0.941196,1
3,0.668623,1
4,0.982474,1


## 5). Save a model manifest (versioning & reproducibility)

In [8]:
manifest = {
    "run_id": run_id,
    "artifact": artifact_path.name,
    "features_order": FEATURES_ORDER,
    "dtypes": {k: str(v) for k, v in DTYPES.items()},
    "sklearn_version": __import__("sklearn").__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__
}
MANIFEST_PATH = MODELS_DIR / f"model_manifest_{run_id}.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
print("Wrote manifest:", MANIFEST_PATH)


Wrote manifest: models/model_manifest_logreg_tuned_seed42_1761367619.json


## 6) Create a reusable serve module (src/serve.py)

In [9]:
%%writefile src/serve.py
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Default paths (override when importing if needed)
MODELS_DIR = Path("models")

# These should match Notebook 09 definitions
FEATURES_ORDER = [
    "age","sex","cp","trestbps","chol","fbs","restecg",
    "thalach","exang","oldpeak","slope","ca","thal"
]
DTYPES = {
    "age": float, "sex": int, "cp": int, "trestbps": float, "chol": float,
    "fbs": int, "restecg": int, "thalach": float, "exang": int,
    "oldpeak": float, "slope": int, "ca": float, "thal": int
}

def load_model(run_id: str) -> object:
    path = MODELS_DIR / f"heart_model_{run_id}.pkl"
    if not path.exists():
        raise FileNotFoundError(f"Model not found: {path}")
    return joblib.load(path)

def validate_input_frame(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in FEATURES_ORDER if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    df = df[FEATURES_ORDER].copy()
    for c, t in DTYPES.items():
        df[c] = df[c].astype(t)
    return df

def predict_proba(model_pipe, df: pd.DataFrame) -> np.ndarray:
    dfv = validate_input_frame(df)
    return model_pipe.predict_proba(dfv)[:, 1]

def predict_label(model_pipe, df: pd.DataFrame, threshold: float = 0.5) -> np.ndarray:
    p = predict_proba(model_pipe, df)
    return (p >= threshold).astype(int)


Overwriting src/serve.py


## 7). Minimal FastAPI service (app/api.py)

In [10]:
%%writefile app/api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, conlist
from typing import List, Optional
import pandas as pd
import joblib
from pathlib import Path
import json

from src.serve import FEATURES_ORDER, DTYPES, predict_proba, predict_label, load_model

app = FastAPI(title="Heart Disease Risk API", version="1.0")

# Load best model by default at startup (from pointer file)
REPORTS_DIR = Path("reports")
MODELS_DIR  = Path("models")
BEST_PATH   = REPORTS_DIR / "best_model.json"

if BEST_PATH.exists():
    best = json.loads(BEST_PATH.read_text())
    RUN_ID = best["best_run_id"]
else:
    # fallback: latest pkl
    candidates = sorted(MODELS_DIR.glob("heart_model_*.pkl"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise RuntimeError("No model artifacts found.")
    RUN_ID = candidates[0].stem.replace("heart_model_", "")

MODEL = load_model(RUN_ID)

class Instance(BaseModel):
    # features must be present (dict-like)
    age: float
    sex: int
    cp: int
    trestbps: float
    chol: float
    fbs: int
    restecg: int
    thalach: float
    exang: int
    oldpeak: float
    slope: int
    ca: float
    thal: int

class BatchRequest(BaseModel):
    instances: List[Instance]
    threshold: Optional[float] = 0.5

@app.get("/health")
def health():
    return {"status": "ok", "run_id": RUN_ID, "features": FEATURES_ORDER}

@app.post("/predict_proba")
def predict_proba_endpoint(payload: BatchRequest):
    df = pd.DataFrame([i.dict() for i in payload.instances])
    try:
        p = predict_proba(MODEL, df)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))
    return {"run_id": RUN_ID, "probabilities": p.tolist()}

@app.post("/predict")
def predict_endpoint(payload: BatchRequest):
    df = pd.DataFrame([i.dict() for i in payload.instances])
    try:
        yhat = predict_label(MODEL, df, threshold=payload.threshold or 0.5)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))
    return {"run_id": RUN_ID, "threshold": payload.threshold or 0.5, "predictions": yhat.tolist()}


Overwriting app/api.py


In [11]:
!pip install -q fastapi uvicorn pydantic

In [12]:
from pyngrok import ngrok
ngrok.kill()
# 👇 Paste your real token between the quotes
ngrok.set_auth_token("34ZtcqRpeDhxTauVy4J1wpYPP43_tjKvSVG2h5kFDaMceN9A")


In [13]:
import subprocess
import time

# Start the server
server = subprocess.Popen(
    ["uvicorn", "app.api:app", "--host", "0.0.0.0", "--port", "8000"]
)

time.sleep(3)  # give it time to start

from pyngrok import ngrok

# Your token must already be set
public_url = ngrok.connect(8000)
public_url


<NgrokTunnel: "https://expectable-maxima-nonconcentric.ngrok-free.dev" -> "http://localhost:8000">

## 8) Simple Streamlit demo

In [18]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 115.5 MB/s eta 0:00:00


In [31]:
%%writefile app/streamlit_app.py
from pathlib import Path
import sys
import os
PROJECT_ROOT = "/content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction"
sys.path.append(PROJECT_ROOT)


from src.serve import load_model, FEATURES_ORDER, predict_proba, predict_label
import streamlit as st
import pandas as pd
import json
# Adjust this path to your real project root


st.set_page_config(page_title="Heart Disease Risk Demo", layout="centered")
st.title("❤️ Heart Disease Risk — Demo")

REPORTS_DIR = Path("reports")
MODELS_DIR  = Path("models")
BEST_PATH   = REPORTS_DIR / "best_model.json"

if BEST_PATH.exists():
    best = json.loads(BEST_PATH.read_text())
    RUN_ID = best["best_run_id"]
else:
    candidates = sorted(MODELS_DIR.glob("heart_model_*.pkl"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        st.error("No model artifacts found.")
        st.stop()
    RUN_ID = candidates[0].stem.replace("heart_model_", "")

model = load_model(RUN_ID)
st.caption(f"Using model run: {RUN_ID}")

# UI inputs
cols = st.columns(2)
inputs = {}
defaults = {"age":57,"sex":1,"cp":2,"trestbps":130,"chol":250,"fbs":0,"restecg":1,"thalach":150,"exang":0,"oldpeak":1.0,"slope":1,"ca":0.0,"thal":2}
for i, feat in enumerate(FEATURES_ORDER):
    with cols[i % 2]:
        if feat in ["sex","cp","fbs","restecg","exang","slope","thal"]:
            inputs[feat] = st.number_input(feat, value=int(defaults.get(feat, 0)))
        else:
            inputs[feat] = st.number_input(feat, value=float(defaults.get(feat, 0.0)))

thr = st.slider("Decision threshold", 0.0, 1.0, 0.5, 0.01)

if st.button("Predict"):
    df = pd.DataFrame([inputs])
    proba = predict_proba(model, df)[0]
    yhat  = int(proba >= thr)
    st.metric("Predicted probability", f"{proba:.3f}")
    st.metric("Predicted label", "Disease" if yhat==1 else "No Disease")
    st.caption("Adjust the threshold to explore sensitivity/specificity tradeoffs.")


Overwriting app/streamlit_app.py


In [34]:
from pyngrok import ngrok

ngrok.kill()  # closes all tunnels in this session


In [35]:
import subprocess, time
from pyngrok import ngrok

# If you haven't already in this runtime:
ngrok.set_auth_token("34ZtcqRpeDhxTauVy4J1wpYPP43_tjKvSVG2h5kFDaMceN9A")

# Start Streamlit
process = subprocess.Popen(
    ["streamlit", "run", "app/streamlit_app.py", "--server.port", "8501", "--server.address", "0.0.0.0"]
)
time.sleep(5)  # give it time to start


In [36]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
public_url



<NgrokTunnel: "https://expectable-maxima-nonconcentric.ngrok-free.dev" -> "http://localhost:8501">

## 9) Batch prediction utility (CSV -> CSV)

In [37]:
def batch_predict_csv(input_csv: str, output_csv: str, threshold: float = 0.5):
    df = pd.read_csv(input_csv)
    proba = predict_proba_frame(pipe, df)
    pred  = (proba >= threshold).astype(int)
    out = df.copy()
    out["proba"] = proba
    out["pred"] = pred
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_csv, index=False)
    print("Wrote:", output_csv)

# Example (uncomment, provide your csv path):
# batch_predict_csv("data/external/new_patients.csv", "reports/new_patients_scored.csv", 0.5)


## 10) ONNX export (if desired)

In [39]:
!pip install -q skl2onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.9 MB/s eta 0:00:00


In [40]:
# ⚠️ Many sklearn pipelines convert; custom transformers can block conversion.

try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    # crude example: pipeline must take a numeric array; if your pipe expects DataFrame, conversion may need extra work
    initial_type = [("float_input", FloatTensorType([None, len(FEATURES_ORDER)]))]
    onnx_model = convert_sklearn(pipe, initial_types=initial_type)
    onnx_path = MODELS_DIR / f"heart_model_{run_id}.onnx"
    with open(onnx_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    print("Exported ONNX:", onnx_path)
except Exception as e:
    print("ONNX export skipped/failed:", e)


ONNX export skipped/failed: Unable to find column name 'age' among names ['variable']. Make sure the input names specified with parameter initial_types fits the column names specified in the pipeline to convert. This may happen because a ColumnTransformer follows a transformer without any mapped converter in a pipeline.


### ✅ Summary
- Validated `heart_model_<run_id>.pkl` with a canonical input schema
- Exposed `predict_proba` / `predict` helpers in `src/serve.py`
- Built **FastAPI** service (`/health`, `/predict_proba`, `/predict`)
- Built **Streamlit** demo for interactive testing
- Wrote a **model manifest** with versions & schema

### 🔜 Next
- Containerize (Dockerfile) and deploy (e.g., Cloud Run, Azure App Service)
- Add auth, request logging, and rate-limiting for production APIs
- Monitor drift and re-train on schedule via CI/CD
